# DLM Behavior Playground

This notebook is a playground for analyzing diffusion language model outputs and behavior. It is not trying to optimize a training objective or force a single conclusion.

The goal is to give you a workspace where you can:

1. Run the same prompts under different DLM denoising schedules.
2. Inspect raw completions without hiding weird failure modes.
3. Quantify obvious behaviors like repetition, special-token leakage, prompt drift, answer instability, and continuation drift.
4. Compare base prompts, counterfactual edits, paraphrases, adversarial variants, and open-ended prompts.
5. Project prompt and completion representations into 2D so you can look for latent-space structure.
6. Save the resulting tables for later mechanistic-interpretability or behavioral analysis.

Use the automated metrics as triage, not truth. The point is to surface patterns worth investigating, then let you decide what deserves a real experiment.


## Mental Model

Autoregressive LMs decode left to right. Diffusion language models instead refine masked/noisy token blocks over multiple denoising steps. That makes the sampling schedule part of the behavior you are studying.

For this notebook, think in three layers:

- **Prompt behavior:** Does the model follow the task, preserve key variables, and answer consistently?
- **Generation behavior:** Does changing `steps`, `block_length`, `threshold`, or `temperature` produce cleaner reasoning, more drift, or more repetition?
- **Representation behavior:** If the model exposes hidden states, do prompts and completions cluster by task, schedule, answer, or failure type?

This notebook mostly uses black-box and gray-box probes. It does not automatically expose the true diffusion trajectory unless the remote model implementation returns intermediate states. Where we use hidden states, they are token-level model representations, not guaranteed denoising-step latents.


In [ ]:
# Run this first in Colab, then restart the runtime before loading the model.
# Efficient-DLM remote code currently expects Transformers 4.x APIs.
%pip install -U "transformers>=4.57.1,<5" accelerate datasets pandas matplotlib packaging plotly

## Configuration

Default model: `nvidia/Efficient-DLM-4B`.

Other DLM-like models you can try if you have enough VRAM and trust their remote code:

- `nvidia/Efficient-DLM-8B`
- `Dream-org/Dream-v0-Instruct-7B`
- `GSAI-ML/LLaDA-8B-Instruct`

Important toggles:

- `RUN_GENERATION`: actually call the model.
- `RUN_EMBEDDING_PROBES`: run input-embedding visualizations. These are lighter than hidden-state probes.
- `RUN_LATENT_PROBES`: run hidden-state probes. These are optional because model APIs differ and they use more memory.
- `LOW_VRAM_MODE`: automatically enabled below 10 GB of GPU memory.
- `USE_ACCELERATE_DEVICE_MAP`: load the model with Accelerate CPU offload instead of moving the whole model onto the GPU.
- `SAVE_RESULTS`: write CSV/JSONL artifacts under `dlm_runs/`.


In [ ]:
import json
import math
import os
import re
import time
from datetime import datetime
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", str(Path("/tmp/matplotlib")))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

try:
    from IPython.display import display
except Exception:
    display = print

MODEL_ID = "nvidia/Efficient-DLM-4B"
MODEL_REVISION = "d28c3167e0972d9c08eb92e03cb40a42d14690d5"
HF_CACHE_DIR = Path("hf_cache")
RUN_DIR = Path("dlm_runs")
HF_CACHE_DIR.mkdir(exist_ok=True)
RUN_DIR.mkdir(exist_ok=True)

os.environ.setdefault("HF_HOME", str(HF_CACHE_DIR.resolve()))
os.environ.setdefault("HF_DATASETS_CACHE", str((HF_CACHE_DIR / "datasets").resolve()))

RUN_GENERATION = True
SAVE_RESULTS = True

SEED = 42

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if device == "cuda" else torch.float32

GPU_TOTAL_GB = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3) if device == "cuda" else 0.0
LOW_VRAM_MODE = device == "cuda" and GPU_TOTAL_GB < 10

# A 4B bf16 model usually does not fit fully on a 6 GB GPU. Device-map loading
# lets Accelerate keep part of the model on CPU/disk instead of calling model.to("cuda").
USE_ACCELERATE_DEVICE_MAP = LOW_VRAM_MODE
FORCE_CPU_MODEL = False
GPU_MEMORY_FRACTION_FOR_MODEL = 0.28
MIN_FREE_CUDA_GB_BEFORE_LOAD = 1.0
CPU_OFFLOAD_DIR = RUN_DIR / "offload"
CPU_OFFLOAD_DIR.mkdir(exist_ok=True)

# Input-embedding probes are lighter than hidden-state probes because they do not run a full forward pass.
RUN_EMBEDDING_PROBES = RUN_GENERATION
# Hidden-state probes are useful but memory-heavy. Enable manually after generation works.
RUN_LATENT_PROBES = RUN_GENERATION and device == "cuda" and not LOW_VRAM_MODE
MAX_NEW_TOKENS = 64 if LOW_VRAM_MODE else 160

# Optional manual overrides:
# RUN_EMBEDDING_PROBES = True
# RUN_LATENT_PROBES = True
# MAX_NEW_TOKENS = 160
# USE_ACCELERATE_DEVICE_MAP = True
# FORCE_CPU_MODEL = True
# GPU_MEMORY_FRACTION_FOR_MODEL = 0.20

torch.manual_seed(SEED)
if device == "cuda":
    torch.cuda.manual_seed_all(SEED)
    torch.cuda.empty_cache()

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.grid"] = True
pd.set_option("display.max_colwidth", 180)

print("device:", device)
print("dtype:", dtype)
print("gpu total GB:", round(GPU_TOTAL_GB, 2) if GPU_TOTAL_GB else "n/a")
print("low VRAM mode:", LOW_VRAM_MODE)
print("device-map offload:", USE_ACCELERATE_DEVICE_MAP)
print("force CPU model:", FORCE_CPU_MODEL)
print("gpu memory fraction for model:", GPU_MEMORY_FRACTION_FOR_MODEL)
print("max new tokens:", MAX_NEW_TOKENS)
print("embedding probes:", RUN_EMBEDDING_PROBES)
print("latent probes:", RUN_LATENT_PROBES)
print("model:", MODEL_ID)
print("run dir:", RUN_DIR.resolve())


## Load Model

Efficient-DLM is loaded with `AutoModel` because its diffusion generation method lives in remote model code. The notebook checks for a couple of Transformers APIs that this pinned revision expects.

If you only want to inspect the notebook structure without model calls, set `RUN_GENERATION = False`; analysis cells will remain empty until real generations exist.

### Low-VRAM Loading

A 4B bf16 model usually needs more than 6 GB of GPU memory if loaded entirely on CUDA. On small GPUs this notebook automatically enables `LOW_VRAM_MODE`, reduces `MAX_NEW_TOKENS`, disables hidden-state probes, and loads with Accelerate CPU offload. The GPU memory cap is intentionally conservative. If CUDA has less than 1 GB free before loading, the notebook falls back to CPU-only loading. If you already hit an OOM, restart the kernel before rerunning so the partially allocated model is released.

For a 5-6 GB GPU, the realistic options are:

- Keep `RUN_GENERATION=False` only when you want to skip model calls; analysis cells will stay empty until real generations exist.
- Try partial offload with `GPU_MEMORY_FRACTION_FOR_MODEL = 0.20`.
- Set `FORCE_CPU_MODEL=True` to avoid CUDA OOM, accepting very slow generation.
- Use a smaller DLM/checkpoint if you want interactive sweeps.



In [ ]:
model = None
tokenizer = None
config = None

if RUN_GENERATION or RUN_EMBEDDING_PROBES or RUN_LATENT_PROBES:
    import transformers
    from packaging import version
    from transformers import AutoConfig, AutoModel, AutoTokenizer
    from transformers import cache_utils
    import transformers.integrations as hf_integrations

    MIN_TRANSFORMERS = "4.57.1"
    MAX_TRANSFORMERS_MAJOR = 5
    print("transformers:", transformers.__version__)

    missing_api = []
    if not hasattr(cache_utils, "SlidingWindowCache"):
        missing_api.append("transformers.cache_utils.SlidingWindowCache")
    if not hasattr(hf_integrations, "use_kernel_forward_from_hub"):
        missing_api.append("transformers.integrations.use_kernel_forward_from_hub")

    current_version = version.parse(transformers.__version__)
    if current_version < version.parse(MIN_TRANSFORMERS) or current_version.major >= MAX_TRANSFORMERS_MAJOR or missing_api:
        raise ImportError(
            "Efficient-DLM cannot load with this Transformers install. "
            f"Found transformers=={transformers.__version__}; missing APIs: {missing_api}. "
            "Run the install cell, restart the kernel/runtime, then rerun from the top."
        )

    if device != "cuda":
        print("CUDA is not available. A 4B DLM on CPU is usually impractical.")
        print("Set RUN_GENERATION=False if you only want the analysis scaffold.")
    elif LOW_VRAM_MODE:
        print(
            f"Detected a {GPU_TOTAL_GB:.2f} GB GPU. Loading with CPU offload and reduced defaults. "
            "Generation will be slower than full-GPU loading."
        )

    load_kwargs = dict(trust_remote_code=True, cache_dir=str(HF_CACHE_DIR))
    if MODEL_REVISION:
        load_kwargs["revision"] = MODEL_REVISION

    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, **load_kwargs)
    if tokenizer.pad_token_id is None and tokenizer.eos_token_id is not None:
        tokenizer.pad_token = tokenizer.eos_token

    config = AutoConfig.from_pretrained(MODEL_ID, **load_kwargs)
    if getattr(config, "pad_token_id", None) is None:
        config.pad_token_id = tokenizer.pad_token_id
    if getattr(config, "eos_token_id", None) is None:
        config.eos_token_id = tokenizer.eos_token_id
    if getattr(config, "bos_token_id", None) is None:
        config.bos_token_id = tokenizer.bos_token_id
    if getattr(config, "vocab_size", None) is None:
        config.vocab_size = len(tokenizer)

    model_load_kwargs = dict(
        config=config,
        torch_dtype=dtype,
        low_cpu_mem_usage=True,
    )

    if FORCE_CPU_MODEL:
        model_load_kwargs.update(device_map={"": "cpu"})
        print("FORCE_CPU_MODEL=True; loading the model on CPU. This is slow but avoids CUDA OOM during load.")
    elif USE_ACCELERATE_DEVICE_MAP:
        if device == "cuda":
            free_bytes, total_bytes = torch.cuda.mem_get_info(0)
            free_gb = free_bytes / (1024 ** 3)
            total_gb = total_bytes / (1024 ** 3)
            print("CUDA free GB before model load:", round(free_gb, 2))

            if free_gb < MIN_FREE_CUDA_GB_BEFORE_LOAD:
                model_load_kwargs.update(device_map={"": "cpu"})
                print(
                    "CUDA has too little free memory for a safe partial load. "
                    "Loading on CPU instead. Restart the kernel if this follows an OOM."
                )
            else:
                target_gb = min(total_gb * GPU_MEMORY_FRACTION_FOR_MODEL, free_gb * 0.60)
                target_mib = max(512, int(target_gb * 1024))
                model_load_kwargs.update(
                    device_map="auto",
                    max_memory={0: f"{target_mib}MiB", "cpu": "48GiB"},
                    offload_folder=str(CPU_OFFLOAD_DIR),
                    offload_state_dict=True,
                )
                print("max GPU memory for model:", model_load_kwargs["max_memory"][0])
        else:
            model_load_kwargs.update(device_map={"": "cpu"})

    try:
        model = AutoModel.from_pretrained(
            MODEL_ID,
            **model_load_kwargs,
            **load_kwargs,
        )
        if not USE_ACCELERATE_DEVICE_MAP and not FORCE_CPU_MODEL:
            model = model.to(device)
        model = model.eval()
        print("loaded", MODEL_ID)
        if hasattr(model, "hf_device_map"):
            print("device map:", model.hf_device_map)
    except torch.cuda.OutOfMemoryError as exc:
        if device == "cuda":
            torch.cuda.empty_cache()
        raise RuntimeError(
            "CUDA OOM while loading the model. Restart the kernel to release the partially loaded model, "
            "then rerun. On a ~6 GB GPU, keep RUN_LATENT_PROBES=False and either lower "
            "GPU_MEMORY_FRACTION_FOR_MODEL to 0.20 or set FORCE_CPU_MODEL=True."
        ) from exc
else:
    print("Skipping model load.")


## Prompt Suite

Prompts are grouped by behavior type rather than by a single benchmark. Add your own rows here.

Suggested categories:

- `arithmetic`: easy tasks where the expected answer is known.
- `counterfactual`: one input variable changes while most text stays fixed.
- `paraphrase`: same task with different surface form.
- `instruction_following`: checks whether the model stays inside format constraints.
- `ambiguous`: useful for studying assumptions and uncertainty.
- `open_reasoning`: no single answer; inspect reasoning style and drift.

Columns:

- `case_id`: stable identifier.
- `family`: group related variants.
- `variant`: base, counterfactual, paraphrase, etc.
- `expected`: optional answer string or number.
- `anchor_terms`: optional terms that should be preserved or used.


In [ ]:
PROMPTS = [
    {
        "case_id": "price_base",
        "family": "price_intervention",
        "variant": "base",
        "category": "arithmetic",
        "prompt": "A notebook costs $4. Maya buys 3 notebooks and pays with a $20 bill. How much change does she get? Show concise reasoning.",
        "expected": "8",
        "anchor_terms": ["notebook", "$4", "3", "$20", "change"],
    },
    {
        "case_id": "price_cf",
        "family": "price_intervention",
        "variant": "counterfactual",
        "category": "counterfactual",
        "prompt": "A notebook costs $5. Maya buys 3 notebooks and pays with a $20 bill. How much change does she get? Show concise reasoning.",
        "expected": "5",
        "anchor_terms": ["notebook", "$5", "3", "$20", "change"],
    },
    {
        "case_id": "apples_base",
        "family": "count_intervention",
        "variant": "base",
        "category": "arithmetic",
        "prompt": "There are 18 apples. 6 are eaten. The rest are split equally among 4 children. How many apples does each child get? Show concise reasoning.",
        "expected": "3",
        "anchor_terms": ["18", "6", "4", "apples", "children"],
    },
    {
        "case_id": "apples_cf",
        "family": "count_intervention",
        "variant": "counterfactual",
        "category": "counterfactual",
        "prompt": "There are 18 apples. 2 are eaten. The rest are split equally among 4 children. How many apples does each child get? Show concise reasoning.",
        "expected": "4",
        "anchor_terms": ["18", "2", "4", "apples", "children"],
    },
    {
        "case_id": "format_json",
        "family": "format_following",
        "variant": "strict",
        "category": "instruction_following",
        "prompt": "Return only valid JSON with keys answer and reason. Question: if a train leaves at 3pm and arrives 2 hours later, what hour does it arrive?",
        "expected": "5",
        "anchor_terms": ["JSON", "answer", "reason", "3pm", "2"],
    },
    {
        "case_id": "ambiguous_bat",
        "family": "ambiguity",
        "variant": "base",
        "category": "ambiguous",
        "prompt": "Explain what a bat is in one sentence. If the word is ambiguous, mention both likely meanings.",
        "expected": None,
        "anchor_terms": ["bat", "ambiguous", "meanings"],
    },
    {
        "case_id": "symbolic_swap",
        "family": "symbolic_reasoning",
        "variant": "base",
        "category": "symbolic",
        "prompt": "Let A mean double and B mean add three. Starting from 5, apply A then B. What is the result? Show concise reasoning.",
        "expected": "13",
        "anchor_terms": ["A", "double", "B", "add", "5"],
    },
]


FINAL_ANSWER_INSTRUCTION = """

Answer formatting requirement:
- Show concise reasoning.
- End with exactly one final line: Final answer: \boxed{ANSWER}
- Do not write anything after the boxed final answer.
""".strip()


def should_require_boxed_answer(prompt_row):
    if prompt_row.get("expected") is None:
        return False
    # Keep strict format-following prompts free to obey their own format.
    if prompt_row.get("category") == "instruction_following":
        return False
    return True


def build_generation_prompt(prompt_row):
    prompt = prompt_row["prompt"]
    if should_require_boxed_answer(prompt_row):
        return prompt.rstrip() + "\n\n" + FINAL_ANSWER_INSTRUCTION
    return prompt


SCHEDULES = [
    {"name": "fast", "steps": 32, "block_length": 32, "threshold": 0.8, "temperature": 0.2},
    {"name": "balanced", "steps": 64, "block_length": 32, "threshold": 0.9, "temperature": 0.2},
    {"name": "deep", "steps": 128, "block_length": 32, "threshold": 0.9, "temperature": 0.2},
    {"name": "exploratory", "steps": 128, "block_length": 32, "threshold": 0.7, "temperature": 0.7},
]

prompt_preview = pd.DataFrame(PROMPTS)[["case_id", "family", "variant", "category", "expected", "prompt"]].copy()
prompt_preview["generation_prompt"] = [build_generation_prompt(row) for row in PROMPTS]
prompt_preview


## Generation Harness

This cell keeps the model-specific generation controls in one place. For Efficient-DLM, `steps` must be compatible with the number of generated blocks, so `_resolve_generation_constraints` rounds it up when needed.

The raw completion is preserved. A cleaned completion is also created later for metrics and plotting. Do not rely only on the cleaned text when studying weird model behavior.


In [ ]:
def _resolve_generation_constraints(schedule, max_new_tokens):
    block_length = int(schedule["block_length"])
    if block_length <= 0:
        raise ValueError(f"block_length must be > 0, got {block_length}")

    effective_max_new_tokens = int(max_new_tokens)
    if effective_max_new_tokens < block_length:
        effective_max_new_tokens = block_length

    remainder = effective_max_new_tokens % block_length
    if remainder != 0:
        effective_max_new_tokens -= remainder

    num_blocks = max(1, effective_max_new_tokens // block_length)
    requested_steps = int(schedule["steps"])
    effective_steps = requested_steps
    if requested_steps % num_blocks != 0:
        effective_steps = ((requested_steps + num_blocks - 1) // num_blocks) * num_blocks

    return {
        "block_length": block_length,
        "num_blocks": num_blocks,
        "requested_steps": requested_steps,
        "effective_steps": effective_steps,
        "effective_max_new_tokens": effective_max_new_tokens,
    }


def get_model_input_device():
    if model is None:
        return device
    if hasattr(model, "hf_device_map"):
        mapped_devices = {str(v) for v in model.hf_device_map.values()}
        if any(v.startswith("cuda") or v == "0" for v in mapped_devices):
            return "cuda"
        return "cpu"
    try:
        return next(model.parameters()).device
    except StopIteration:
        return device


def generate_dlm(prompt, schedule):
    if model is None or tokenizer is None:
        raise RuntimeError("Model/tokenizer are not loaded.")

    settings = _resolve_generation_constraints(schedule, MAX_NEW_TOKENS)
    if settings["effective_steps"] != settings["requested_steps"]:
        print(
            f"adjusting steps {settings['requested_steps']} -> {settings['effective_steps']} "
            f"for block_length={settings['block_length']} and max_new_tokens={settings['effective_max_new_tokens']}"
        )

    input_device = get_model_input_device()
    inputs = tokenizer(prompt, return_tensors="pt").to(input_device)
    t0 = time.time()
    with torch.inference_mode():
        out = model.generate(
            inputs.input_ids,
            max_new_tokens=settings["effective_max_new_tokens"],
            steps=settings["effective_steps"],
            block_length=settings["block_length"],
            shift_logits=False,
            temperature=schedule["temperature"],
            threshold=schedule["threshold"],
        )
    elapsed = time.time() - t0

    if isinstance(out, tuple):
        out_ids, nfe = out
    else:
        out_ids, nfe = out, None

    completion_ids = out_ids[:, inputs.input_ids.shape[-1]:]
    completion = tokenizer.batch_decode(completion_ids, skip_special_tokens=False)[0]
    completion_skip_special = tokenizer.batch_decode(completion_ids, skip_special_tokens=True)[0]

    return {
        "completion_raw": completion.strip(),
        "completion_skip_special": completion_skip_special.strip(),
        "nfe": nfe,
        "seconds": elapsed,
        "prompt_tokens": int(inputs.input_ids.shape[-1]),
        "completion_tokens": int(completion_ids.shape[-1]),
        "requested_steps": settings["requested_steps"],
        "steps": settings["effective_steps"],
        "max_new_tokens": settings["effective_max_new_tokens"],
    }


def run_generation_grid(prompts, schedules):
    rows = []
    if not RUN_GENERATION:
        print("RUN_GENERATION=False; skipping model calls.")
        return rows

    for prompt_row in prompts:
        generation_prompt = build_generation_prompt(prompt_row)
        for schedule in schedules:
            print(prompt_row["case_id"], schedule["name"])
            result = generate_dlm(generation_prompt, schedule)
            rows.append({**prompt_row, "generation_prompt": generation_prompt, **schedule, **result})
    return rows

rows = run_generation_grid(PROMPTS, SCHEDULES)
print("rows:", len(rows))

PROMPT_COLUMNS = sorted({key for row in PROMPTS for key in row.keys()})
SCHEDULE_COLUMNS = sorted({key for row in SCHEDULES for key in row.keys()})
RESULT_COLUMNS = [
    "completion_raw", "completion_skip_special", "nfe", "seconds", "prompt_tokens",
    "completion_tokens", "requested_steps", "steps", "max_new_tokens",
]
DF_RAW_COLUMNS = PROMPT_COLUMNS + [c for c in SCHEDULE_COLUMNS + RESULT_COLUMNS if c not in PROMPT_COLUMNS]

df_raw = pd.DataFrame(rows, columns=DF_RAW_COLUMNS)
if df_raw.empty:
    print("No generation rows yet. Run generation to populate df_raw and downstream analysis cells.")
df_raw.head()


## Output Cleaning And Diagnostics

The notebook keeps both raw and cleaned completions.

Useful things to look for:

- Special-token leakage such as `<|fim_pad|>`.
- Repetitive tails.
- The model answering correctly, then drifting into unrelated tasks.
- Different schedules producing different final numbers for the same prompt.
- Counterfactual variants that should change but do not.

These metrics are intentionally simple. They are designed to help you sort and inspect examples, not replace manual reading.

Accuracy is strict now. The notebook parses the final answer from one of these sources, in this order:

1. `\boxed{...}`
2. JSON object with an `answer` key
3. an explicit `Final answer:` or `Answer:` line

If none of those exists, the answer is marked as missing instead of guessing from the last number in the completion. Set `RUN_LOCAL_LLM_JUDGE=True` if you want a local judge model, such as Ollama, to grade final-answer correctness.


In [ ]:
STOPWORDS = set("a an the and or to of in for with is are does do did how many much show concise reasoning then rest per by among from only valid keys question what if it this that".split())
SPECIAL_TOKEN_RE = re.compile(r"<\|[^>]+\|>")
DRIFT_MARKERS = ["input:", "question:", "problem:", "write a python", "prove that", "output:"]


def normalize_text(s):
    return re.sub(r"\s+", " ", str(s).lower()).strip()


def strip_special_tokens(s):
    return SPECIAL_TOKEN_RE.sub(" ", str(s))


def truncate_at_drift_marker(s):
    lower = str(s).lower()
    cuts = [lower.find(marker) for marker in DRIFT_MARKERS if lower.find(marker) > 0]
    if not cuts:
        return str(s)
    return str(s)[:min(cuts)].strip()


def clean_for_scoring(s):
    return normalize_text(truncate_at_drift_marker(strip_special_tokens(s)))


def extract_numbers(s):
    return re.findall(r"-?\d+(?:\.\d+)?", str(s))


def last_number(s):
    nums = extract_numbers(s)
    return nums[-1] if nums else None


ANSWER_JSON_RE = re.compile(r"\{.*?\}", re.DOTALL)
BOXED_RE = re.compile(r"\\boxed\s*\{([^{}]+)\}")
FINAL_ANSWER_RE = re.compile(r"(?:final\s+answer|answer)\s*[:：]\s*(.+)", re.IGNORECASE)

RUN_LOCAL_LLM_JUDGE = False
LOCAL_LLM_JUDGE_BACKEND = "ollama"  # "ollama" or "transformers"
OLLAMA_JUDGE_MODEL = "llama3.2"
OLLAMA_JUDGE_TIMEOUT_SECONDS = 30
TRANSFORMERS_JUDGE_MODEL = ""  # local path or already-downloaded HF id


def expected_is_missing(expected):
    return expected is None or (isinstance(expected, float) and math.isnan(expected)) or str(expected).strip() == ""


def normalize_answer_value(s):
    if expected_is_missing(s):
        return None
    text = str(s).strip().lower()
    text = re.sub(r"\\boxed\s*\{([^{}]+)\}", r"\1", text)
    text = text.strip("`*_ \n\t.,;:!?")
    text = re.sub(r"\s+", " ", text)
    # Normalize simple numeric answers like "$8", "8.0", and "8 dollars".
    number_match = re.search(r"-?\d+(?:\.\d+)?", text.replace(",", ""))
    if number_match:
        num = number_match.group(0)
        try:
            f = float(num)
            if f.is_integer():
                return str(int(f))
            return str(f)
        except Exception:
            return num
    return text


def extract_json_answer(text):
    for match in ANSWER_JSON_RE.finditer(str(text)):
        try:
            obj = json.loads(match.group(0))
        except Exception:
            continue
        if isinstance(obj, dict) and "answer" in obj:
            return str(obj["answer"]).strip(), "json_answer"
    return None, None


def extract_final_answer(completion):
    text = str(completion).strip()
    boxed = BOXED_RE.findall(text)
    if boxed:
        return boxed[-1].strip(), "boxed"

    json_answer, source = extract_json_answer(text)
    if json_answer is not None:
        return json_answer, source

    lines = [line.strip() for line in text.splitlines() if line.strip()]
    for line in reversed(lines):
        m = FINAL_ANSWER_RE.search(line)
        if m:
            answer = m.group(1).strip()
            answer = SPECIAL_TOKEN_RE.sub(" ", answer).strip()
            return answer, "final_answer_line"

    return None, "no_final_answer_found"


def exact_final_answer_correct(completion, expected):
    if expected_is_missing(expected):
        return np.nan
    parsed, source = extract_final_answer(completion)
    if parsed is None:
        return 0.0
    return float(normalize_answer_value(parsed) == normalize_answer_value(expected))


def local_llm_judge_answer(prompt, completion, expected):
    if expected_is_missing(expected):
        return {"llm_judge_accuracy": np.nan, "llm_judge_reason": "no expected answer"}
    judge_prompt = f"""
You are grading whether a model's final answer is correct.
Return only compact JSON with keys: correct, final_answer, reason.
correct must be true or false.

Problem:
{prompt}

Expected answer:
{expected}

Model response:
{completion}
""".strip()

    if LOCAL_LLM_JUDGE_BACKEND == "ollama":
        try:
            import requests
            response = requests.post(
                "http://localhost:11434/api/generate",
                json={"model": OLLAMA_JUDGE_MODEL, "prompt": judge_prompt, "stream": False, "format": "json"},
                timeout=OLLAMA_JUDGE_TIMEOUT_SECONDS,
            )
            response.raise_for_status()
            raw = response.json().get("response", "{}")
            data = json.loads(raw)
            return {
                "llm_judge_accuracy": float(bool(data.get("correct"))),
                "llm_judge_final_answer": data.get("final_answer"),
                "llm_judge_reason": data.get("reason"),
            }
        except Exception as exc:
            return {"llm_judge_accuracy": np.nan, "llm_judge_reason": f"ollama judge failed: {exc!r}"}

    if LOCAL_LLM_JUDGE_BACKEND == "transformers":
        try:
            from transformers import AutoModelForCausalLM, AutoTokenizer
            judge_tokenizer = AutoTokenizer.from_pretrained(TRANSFORMERS_JUDGE_MODEL, local_files_only=True)
            judge_model = AutoModelForCausalLM.from_pretrained(TRANSFORMERS_JUDGE_MODEL, local_files_only=True, device_map="auto")
            inputs = judge_tokenizer(judge_prompt, return_tensors="pt").to(next(judge_model.parameters()).device)
            with torch.inference_mode():
                out = judge_model.generate(**inputs, max_new_tokens=160, do_sample=False)
            raw = judge_tokenizer.decode(out[0, inputs.input_ids.shape[-1]:], skip_special_tokens=True)
            match = ANSWER_JSON_RE.search(raw)
            data = json.loads(match.group(0) if match else raw)
            return {
                "llm_judge_accuracy": float(bool(data.get("correct"))),
                "llm_judge_final_answer": data.get("final_answer"),
                "llm_judge_reason": data.get("reason"),
            }
        except Exception as exc:
            return {"llm_judge_accuracy": np.nan, "llm_judge_reason": f"transformers judge failed: {exc!r}"}

    return {"llm_judge_accuracy": np.nan, "llm_judge_reason": f"unknown backend {LOCAL_LLM_JUDGE_BACKEND!r}"}


def content_terms(prompt):
    terms = re.findall(r"[a-zA-Z0-9$]+", normalize_text(prompt))
    return [t for t in terms if t not in STOPWORDS and len(t) > 1]


def anchor_score(anchor_terms, prompt, completion):
    if isinstance(anchor_terms, str):
        try:
            anchor_terms = json.loads(anchor_terms)
        except Exception:
            anchor_terms = []
    terms = anchor_terms or content_terms(prompt)
    if not terms:
        return np.nan
    c = normalize_text(completion)
    return sum(normalize_text(t) in c for t in terms) / len(terms)


def repeated_ngram_rate(text, n=3):
    toks = re.findall(r"\w+", normalize_text(text))
    if len(toks) < n + 1:
        return 0.0
    grams = [tuple(toks[i:i+n]) for i in range(len(toks)-n+1)]
    return 1.0 - (len(set(grams)) / len(grams))


def special_token_count(text):
    return len(SPECIAL_TOKEN_RE.findall(str(text)))


def drift_marker_count(text):
    lower = str(text).lower()
    return sum(lower.count(marker) for marker in DRIFT_MARKERS)


def word_count(text):
    return len(re.findall(r"\w+", str(text)))


def add_diagnostics(df):
    if df.empty:
        return df
    out = df.copy()
    out["completion_clean"] = out["completion_raw"].map(clean_for_scoring)
    out["raw_word_count"] = out["completion_raw"].map(word_count)
    out["clean_word_count"] = out["completion_clean"].map(word_count)
    out["special_token_count"] = out["completion_raw"].map(special_token_count)
    out["drift_marker_count"] = out["completion_raw"].map(drift_marker_count)
    out["redundancy_raw"] = out["completion_raw"].map(repeated_ngram_rate)
    out["redundancy_clean"] = out["completion_clean"].map(repeated_ngram_rate)
    out["last_number_raw"] = out["completion_raw"].map(last_number)
    out["last_number_clean"] = out["completion_clean"].map(last_number)
    parsed_answers = out["completion_clean"].map(extract_final_answer)
    out["parsed_final_answer"] = parsed_answers.map(lambda x: x[0])
    out["final_answer_source"] = parsed_answers.map(lambda x: x[1])
    out["final_answer_accuracy"] = out.apply(lambda r: exact_final_answer_correct(r["completion_clean"], r.get("expected")), axis=1)
    # Backward-compatible column name for older cells/exports; value is now strict final-answer accuracy.
    out["answer_seen_clean_tail"] = out["final_answer_accuracy"]
    if RUN_LOCAL_LLM_JUDGE:
        judge_rows = out.apply(lambda r: local_llm_judge_answer(r["prompt"], r["completion_clean"], r.get("expected")), axis=1)
        judge_df = pd.DataFrame(judge_rows.tolist(), index=out.index)
        out = pd.concat([out, judge_df], axis=1)
        out["accuracy_for_plots"] = out["llm_judge_accuracy"].combine_first(out["final_answer_accuracy"])
    else:
        out["accuracy_for_plots"] = out["final_answer_accuracy"]
    out["anchor_score_clean"] = out.apply(lambda r: anchor_score(r.get("anchor_terms"), r["prompt"], r["completion_clean"]), axis=1)
    out["tail_drift_words"] = (out["raw_word_count"] - out["clean_word_count"]).clip(lower=0)
    return out


df = add_diagnostics(df_raw)
cols = [
    "case_id", "variant", "name", "steps", "threshold", "temperature", "nfe", "seconds",
    "final_answer_accuracy", "accuracy_for_plots", "parsed_final_answer", "final_answer_source",
    "anchor_score_clean", "special_token_count", "drift_marker_count",
    "redundancy_raw", "tail_drift_words", "last_number_clean",
]
df[cols] if len(df) else df


## Raw Output Browser

Use this cell before trusting any plot. A lot of interesting DLM behavior is visible only in the raw text: padding-token leakage, unwanted continuation, code/data contamination, and schedule-specific collapse.


In [ ]:
def show_outputs(df, case_id=None, schedule=None, max_chars=1400, raw=True):
    if df.empty:
        print("No generation rows to inspect yet.")
        return
    view = df.copy()
    if case_id is not None:
        view = view[view["case_id"].eq(case_id)]
    if schedule is not None:
        view = view[view["name"].eq(schedule)]

    for _, row in view.sort_values(["case_id", "name"]).iterrows():
        print("=" * 100)
        print(f"{row['case_id']} | {row['variant']} | {row['name']} | steps={row['steps']} | nfe={row.get('nfe')}")
        print("prompt:", row["prompt"])
        print("expected:", row.get("expected"), "last_clean:", row.get("last_number_clean"))
        print("diagnostics:", {
            "final_answer_accuracy": row.get("accuracy_for_plots"),
            "anchor_score_clean": row.get("anchor_score_clean"),
            "special_token_count": row.get("special_token_count"),
            "drift_marker_count": row.get("drift_marker_count"),
            "redundancy_raw": round(float(row.get("redundancy_raw", 0)), 3),
            "tail_drift_words": row.get("tail_drift_words"),
        })
        text = row["completion_raw"] if raw else row["completion_clean"]
        print("completion:", str(text)[:max_chars])

show_outputs(df, max_chars=900, raw=True)


## Schedule-Level Views

These plots answer basic questions:

- Which schedules are slower?
- Which schedules leak special tokens?
- Which schedules produce long or repetitive tails?
- Which schedules preserve prompt anchors?
- Which schedules are unstable on answer-like tasks?

For real experiments, increase the prompt suite before drawing conclusions.


In [ ]:
def summarize_by_schedule(df):
    if df.empty:
        return pd.DataFrame()
    metric_cols = [
        "accuracy_for_plots", "anchor_score_clean", "special_token_count", "drift_marker_count",
        "redundancy_raw", "redundancy_clean", "tail_drift_words", "raw_word_count", "clean_word_count",
        "seconds", "nfe",
    ]
    return df.groupby("name")[metric_cols].mean(numeric_only=True).reset_index()

summary = summarize_by_schedule(df)
display(summary)

if len(summary):
    fig, ax = plt.subplots(2, 3, figsize=(16, 8))
    plots = [
        ("seconds", "Mean latency"),
        ("nfe", "Mean NFE"),
        ("accuracy_for_plots", "Strict final-answer accuracy"),
        ("anchor_score_clean", "Prompt anchor retention"),
        ("redundancy_raw", "Raw repetition"),
        ("tail_drift_words", "Words removed as drift tail"),
    ]
    for axis, (col, title) in zip(ax.ravel(), plots):
        if col in summary:
            axis.bar(summary["name"], summary[col])
            axis.set_title(title)
            axis.tick_params(axis="x", rotation=20)
    plt.tight_layout()


## Case By Schedule Matrix

Heatmaps are useful for spotting interaction effects. A schedule can look fine on average while failing one prompt family badly.


In [ ]:
def plot_metric_matrix(df, metric, title=None):
    if df.empty or metric not in df:
        print(f"No data for {metric}")
        return
    pivot = df.pivot_table(index="case_id", columns="name", values=metric, aggfunc="mean")
    display(pivot)
    fig, axis = plt.subplots(figsize=(max(8, 1.2 * len(pivot.columns)), max(4, 0.45 * len(pivot))))
    im = axis.imshow(pivot.fillna(0).values, aspect="auto")
    axis.set_xticks(range(len(pivot.columns)))
    axis.set_xticklabels(pivot.columns, rotation=30, ha="right")
    axis.set_yticks(range(len(pivot.index)))
    axis.set_yticklabels(pivot.index)
    axis.set_title(title or metric)
    for i in range(len(pivot.index)):
        for j in range(len(pivot.columns)):
            value = pivot.iloc[i, j]
            label = "" if pd.isna(value) else f"{value:.2f}"
            axis.text(j, i, label, ha="center", va="center", color="white" if (not pd.isna(value) and value > pivot.max().max() / 2) else "black")
    fig.colorbar(im, ax=axis)
    plt.tight_layout()

plot_metric_matrix(df, "accuracy_for_plots", "Final-answer accuracy by case and schedule")
plot_metric_matrix(df, "redundancy_raw", "Raw repetition by case and schedule")
plot_metric_matrix(df, "tail_drift_words", "Continuation drift tail by case and schedule")


## Counterfactual And Variant Stability

For families with multiple variants, compare extracted answers across schedules. This is a behavior table for seeing whether the model tracks prompt edits or collapses to the same answer.


In [ ]:
if len(df):
    answer_table = df.pivot_table(
        index=["family", "name"],
        columns="variant",
        values="last_number_clean",
        aggfunc="first",
    ).reset_index()
    display(answer_table)

    families_with_variants = df.groupby("family")["variant"].nunique()
    interesting_families = families_with_variants[families_with_variants > 1].index.tolist()
    print("families with multiple variants:", interesting_families)
else:
    print("No data.")


## Scatter Views

Scatter plots help you find examples worth reading manually. For example, high repetition with low latency may indicate under-denoising, while low repetition with wrong answers may indicate clean but semantically wrong continuations.


In [ ]:
if len(df):
    fig, ax = plt.subplots(1, 2, figsize=(14, 5))

    for name, group in df.groupby("name"):
        ax[0].scatter(group["seconds"], group["redundancy_raw"], label=name, s=70)
    ax[0].set_xlabel("seconds")
    ax[0].set_ylabel("raw repetition")
    ax[0].set_title("Latency vs repetition")
    ax[0].legend()

    for name, group in df.groupby("name"):
        ax[1].scatter(group["anchor_score_clean"], group["tail_drift_words"], label=name, s=70)
    ax[1].set_xlabel("anchor score clean")
    ax[1].set_ylabel("tail drift words")
    ax[1].set_title("Anchor retention vs drift tail")
    ax[1].legend()

    for _, row in df.iterrows():
        ax[1].annotate(row["case_id"], (row["anchor_score_clean"], row["tail_drift_words"]), fontsize=8, alpha=0.75)

    plt.tight_layout()


## Output Comparison And Token Diagnostics

These views focus on generated text behavior rather than model internals:

- schedule small multiples for reading the same prompt across schedules
- token-level heatmaps for special-token leakage, drift markers, repeated tokens, and numeric tokens
- counterfactual prompt/completion difference views
- answer-number traces by token position
- drift and length timelines
- failure taxonomy dashboard


In [ ]:
def schedule_small_multiples(df, case_id=None, max_chars=900):
    if df.empty:
        print("No generation rows for schedule comparison.")
        return
    view = df.copy()
    if case_id is None:
        case_id = view["case_id"].iloc[0]
    view = view[view["case_id"].eq(case_id)].sort_values("name")
    if view.empty:
        print(f"No rows for case_id={case_id!r}")
        return
    print("case_id:", case_id)
    print("prompt:", view.iloc[0]["prompt"])
    for _, row in view.iterrows():
        print("=" * 100)
        print(
            f"{row['name']} | steps={row.get('steps')} | nfe={row.get('nfe')} | "
            f"answer={row.get('accuracy_for_plots')} | anchor={row.get('anchor_score_clean')} | "
            f"drift_words={row.get('tail_drift_words')} | special={row.get('special_token_count')}"
        )
        print(str(row.get("completion_raw", ""))[:max_chars])


def token_diagnostic_table(text):
    tokens = re.findall(r"<\|[^>]+\|>|\$?\d+(?:\.\d+)?|[A-Za-z_]+|\S", str(text))
    rows = []
    seen = {}
    lower_text = str(text).lower()
    for i, tok in enumerate(tokens):
        norm = normalize_text(tok)
        seen[norm] = seen.get(norm, 0) + 1
        rows.append({
            "pos": i,
            "token": tok,
            "is_special": float(bool(SPECIAL_TOKEN_RE.fullmatch(tok))),
            "is_number": float(bool(re.fullmatch(r"\$?\d+(?:\.\d+)?", tok))),
            "is_repeat": float(seen[norm] > 1 and len(norm) > 1),
            "is_drift_marker": float(any(marker.strip(":") == norm for marker in DRIFT_MARKERS)),
        })
    return pd.DataFrame(rows)


def plot_token_heatmap(text, title="Token diagnostics", max_tokens=180):
    table = token_diagnostic_table(text).head(max_tokens)
    if table.empty:
        print("No tokens to plot.")
        return table
    metrics = ["is_special", "is_number", "is_repeat", "is_drift_marker"]
    fig, axis = plt.subplots(figsize=(max(10, len(table) * 0.08), 3.6))
    axis.imshow(table[metrics].T.values, aspect="auto", interpolation="nearest")
    axis.set_yticks(range(len(metrics)))
    axis.set_yticklabels(metrics)
    axis.set_xlabel("token position")
    axis.set_title(title)
    tick_step = max(1, len(table) // 30)
    axis.set_xticks(range(0, len(table), tick_step))
    axis.set_xticklabels(table["pos"].iloc[::tick_step], rotation=0)
    plt.tight_layout()
    display(table.head(80))
    return table


def plot_case_token_heatmaps(df, case_id=None, max_rows=4):
    if df.empty:
        print("No generation rows for token heatmaps.")
        return {}
    view = df.copy()
    if case_id is None:
        case_id = view["case_id"].iloc[0]
    view = view[view["case_id"].eq(case_id)].head(max_rows)
    tables = {}
    for _, row in view.iterrows():
        title = f"{row['case_id']} | {row['name']} token diagnostics"
        tables[(row["case_id"], row["name"])] = plot_token_heatmap(row["completion_raw"], title=title)
    return tables


def diff_words(a, b):
    import difflib
    a_words = re.findall(r"\w+|\S", str(a))
    b_words = re.findall(r"\w+|\S", str(b))
    return " ".join(difflib.ndiff(a_words, b_words))


def counterfactual_difference_view(df, family=None, schedule=None, max_chars=2500):
    if df.empty:
        print("No generation rows for counterfactual diff view.")
        return
    view = df.copy()
    if family is None:
        counts = view.groupby("family")["variant"].nunique()
        multi = counts[counts > 1]
        if multi.empty:
            print("No multi-variant families found.")
            return
        family = multi.index[0]
    view = view[view["family"].eq(family)]
    if schedule is not None:
        view = view[view["name"].eq(schedule)]
    elif "name" in view and not view.empty:
        schedule = view["name"].iloc[0]
        view = view[view["name"].eq(schedule)]
    if view["variant"].nunique() < 2:
        print("Need at least two variants for diff view.")
        return
    rows = view.sort_values("variant").head(2)
    a, b = rows.iloc[0], rows.iloc[1]
    print("family:", family, "schedule:", schedule)
    print("prompt diff:")
    print(diff_words(a["prompt"], b["prompt"])[:max_chars])
    print("\ncompletion diff:")
    print(diff_words(a.get("completion_clean", ""), b.get("completion_clean", ""))[:max_chars])


def plot_number_trace(df, case_id=None):
    if df.empty:
        print("No generation rows for number trace.")
        return pd.DataFrame()
    view = df.copy()
    if case_id is None:
        case_id = view["case_id"].iloc[0]
    view = view[view["case_id"].eq(case_id)]
    rows = []
    for _, row in view.iterrows():
        for m in re.finditer(r"-?\d+(?:\.\d+)?", str(row.get("completion_raw", ""))):
            rows.append({"case_id": row["case_id"], "schedule": row["name"], "char_pos": m.start(), "number": m.group(0)})
    out = pd.DataFrame(rows)
    if out.empty:
        print("No numbers found in completions for", case_id)
        return out
    fig, axis = plt.subplots(figsize=(10, max(3, 0.5 * out["schedule"].nunique())))
    schedules = {name: i for i, name in enumerate(sorted(out["schedule"].unique()))}
    axis.scatter(out["char_pos"], out["schedule"].map(schedules), s=80)
    for _, row in out.iterrows():
        axis.annotate(row["number"], (row["char_pos"], schedules[row["schedule"]]), fontsize=8)
    axis.set_yticks(list(schedules.values()))
    axis.set_yticklabels(list(schedules.keys()))
    axis.set_xlabel("character position in raw completion")
    axis.set_title(f"Number trace: {case_id}")
    plt.tight_layout()
    return out


def plot_drift_timeline(df):
    if df.empty:
        print("No generation rows for drift timeline.")
        return
    metrics = ["raw_word_count", "clean_word_count", "tail_drift_words", "special_token_count", "drift_marker_count", "redundancy_raw"]
    available = [m for m in metrics if m in df]
    if not available:
        print("No drift metrics available.")
        return
    view = df.sort_values(["case_id", "name"]).reset_index(drop=True)
    fig, axes = plt.subplots(len(available), 1, figsize=(14, 2.6 * len(available)), sharex=True)
    if len(available) == 1:
        axes = [axes]
    x = np.arange(len(view))
    labels = (view["case_id"] + ":" + view["name"]).tolist()
    for axis, metric in zip(axes, available):
        axis.bar(x, view[metric].fillna(0))
        axis.set_ylabel(metric)
    axes[-1].set_xticks(x)
    axes[-1].set_xticklabels(labels, rotation=70, ha="right", fontsize=8)
    axes[0].set_title("Length, drift, and repetition timeline")
    plt.tight_layout()


def failure_taxonomy(df):
    if df.empty:
        print("No generation rows for failure taxonomy.")
        return pd.DataFrame()
    out = df[["case_id", "family", "variant", "name"]].copy()
    out["missing_expected_answer"] = df["accuracy_for_plots"].fillna(1).eq(0).astype(float)
    out["special_token_leak"] = df["special_token_count"].fillna(0).gt(0).astype(float)
    out["drift_tail"] = df["tail_drift_words"].fillna(0).gt(0).astype(float)
    out["repetitive"] = df["redundancy_raw"].fillna(0).gt(0.25).astype(float)
    out["low_anchor"] = df["anchor_score_clean"].fillna(1).lt(0.5).astype(float)
    out["failure_score"] = out[["missing_expected_answer", "special_token_leak", "drift_tail", "repetitive", "low_anchor"]].sum(axis=1)
    display(out.sort_values("failure_score", ascending=False))

    metric_cols = ["missing_expected_answer", "special_token_leak", "drift_tail", "repetitive", "low_anchor"]
    fig, axis = plt.subplots(figsize=(10, max(4, 0.35 * len(out))))
    heat = out.sort_values("failure_score", ascending=False)[metric_cols].values
    axis.imshow(heat, aspect="auto", interpolation="nearest")
    axis.set_xticks(range(len(metric_cols)))
    axis.set_xticklabels(metric_cols, rotation=30, ha="right")
    axis.set_yticks(range(len(out)))
    axis.set_yticklabels((out.sort_values("failure_score", ascending=False)["case_id"] + ":" + out.sort_values("failure_score", ascending=False)["name"]).tolist(), fontsize=8)
    axis.set_title("Failure taxonomy heatmap")
    plt.tight_layout()
    return out


if len(df):
    schedule_small_multiples(df)
    token_heatmap_tables = plot_case_token_heatmaps(df)
    counterfactual_difference_view(df)
    number_trace_df = plot_number_trace(df)
    plot_drift_timeline(df)
    failure_taxonomy_df = failure_taxonomy(df)
else:
    print("Skipping output comparison diagnostics; no real generation rows.")


## Token And Text Utilities

Before doing representation plots, inspect tokenization. Many strange behaviors begin with special tokens, prompt formatting, or unexpected token boundaries.


In [ ]:
def inspect_tokenization(text, max_tokens=80):
    if tokenizer is None:
        print("Tokenizer is not loaded.")
        return pd.DataFrame()
    ids = tokenizer(text, return_tensors="pt").input_ids[0].tolist()
    toks = tokenizer.convert_ids_to_tokens(ids)
    table = pd.DataFrame({"pos": range(len(ids)), "token_id": ids, "token": toks})
    display(table.head(max_tokens))
    print("total tokens:", len(ids))
    return table

if tokenizer is not None and len(df):
    inspect_tokenization(df.iloc[0]["prompt"], max_tokens=60)
else:
    print("Tokenizer unavailable or no rows.")


## Input Embedding Space

This section visualizes the model's input embedding space. These plots are lighter than hidden-state probes because they do not run a full transformer forward pass. They only use `model.get_input_embeddings()`.

Useful questions:

- Do prompts cluster by task family before any reasoning happens?
- Do generated completions occupy different embedding regions by schedule?
- Do individual prompt tokens follow a smooth or abrupt path through embedding PCA space?
- Which vocabulary tokens are nearest to specific terms under the model's embedding geometry?

Caution: input embeddings are lexical/subword representations. They are not the same thing as reasoning states, but they are a useful baseline before looking at hidden layers.


In [ ]:
def pca_2d_embedding(x):
    x = np.asarray(x, dtype=np.float32)
    if x.ndim != 2 or len(x) == 0:
        return np.zeros((0, 2), dtype=np.float32)
    x = x - x.mean(axis=0, keepdims=True)
    u, s, vt = np.linalg.svd(x, full_matrices=False)
    if vt.shape[0] == 1:
        return np.column_stack([x @ vt[0], np.zeros(len(x))])
    return x @ vt[:2].T


def get_input_embedding_layer(verbose=True):
    if model is None:
        raise RuntimeError("Model is not loaded.")

    # Standard Transformers path. EfficientDLM currently raises NotImplementedError here,
    # so we fall back to searching the module tree below.
    try:
        emb = model.get_input_embeddings()
        if emb is not None:
            if verbose:
                print("input embedding layer: model.get_input_embeddings()")
            return emb
    except NotImplementedError:
        pass
    except AttributeError:
        pass

    vocab_size = None
    if tokenizer is not None:
        try:
            vocab_size = len(tokenizer)
        except Exception:
            vocab_size = None
    if vocab_size is None and config is not None:
        vocab_size = getattr(config, "vocab_size", None)

    candidates = []
    for name, module in model.named_modules():
        if isinstance(module, torch.nn.Embedding):
            score = 0
            lname = name.lower()
            if vocab_size is not None and getattr(module, "num_embeddings", None) == vocab_size:
                score += 100
            if any(key in lname for key in ["embed_tokens", "wte", "word_embeddings", "token_embedding", "tok_embeddings"]):
                score += 25
            if "position" in lname or "rotary" in lname:
                score -= 50
            candidates.append((score, name, module))

    if not candidates:
        raise RuntimeError("Could not find any torch.nn.Embedding modules in this model.")

    candidates.sort(key=lambda x: (x[0], getattr(x[2], "num_embeddings", 0)), reverse=True)
    score, name, emb = candidates[0]
    if vocab_size is not None and emb.num_embeddings != vocab_size:
        print(
            "Warning: selected embedding layer does not match tokenizer vocab size:",
            {"layer": name, "num_embeddings": emb.num_embeddings, "vocab_size": vocab_size},
        )
    elif verbose:
        print("input embedding layer:", name, {"num_embeddings": emb.num_embeddings, "dim": emb.embedding_dim})
    return emb


def embedding_device(emb):
    return emb.weight.device


def text_input_embedding_representations(texts, max_length=256, pooling="mean", batch_size=16):
    if model is None or tokenizer is None:
        raise RuntimeError("Model/tokenizer are not loaded.")
    emb = get_input_embedding_layer()
    dev = embedding_device(emb)
    reps = []
    texts = list(texts)
    with torch.inference_mode():
        for start in range(0, len(texts), batch_size):
            batch_texts = texts[start:start + batch_size]
            batch = tokenizer(
                batch_texts,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=max_length,
            ).to(dev)
            token_emb = emb(batch["input_ids"])
            mask = batch["attention_mask"].unsqueeze(-1).to(token_emb.dtype)
            if pooling == "sum":
                pooled = (token_emb * mask).sum(dim=1)
            else:
                pooled = (token_emb * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
            reps.append(pooled.float().cpu().numpy())
    return np.concatenate(reps, axis=0) if reps else np.zeros((0, 2), dtype=np.float32)


def plot_input_embedding_space(df, text_col="prompt", label_col="family", title=None):
    if df.empty:
        print("No data.")
        return pd.DataFrame()
    try:
        reps = text_input_embedding_representations(df[text_col].astype(str).tolist())
    except Exception as exc:
        print("Input embedding probe failed:", repr(exc))
        return pd.DataFrame()

    coords = pca_2d_embedding(reps)
    plot_df = df[["case_id", "name", "family", "variant", "category"]].copy()
    plot_df["x"] = coords[:, 0]
    plot_df["y"] = coords[:, 1]

    fig, axis = plt.subplots(figsize=(9, 6))
    for label, group in plot_df.groupby(label_col):
        axis.scatter(group["x"], group["y"], label=label, s=75)
        for _, row in group.iterrows():
            label_text = row["case_id"] if text_col == "prompt" else f"{row['case_id']}:{row['name']}"
            axis.annotate(label_text, (row["x"], row["y"]), fontsize=8, alpha=0.8)
    axis.set_title(title or f"Input embedding PCA: {text_col}")
    axis.set_xlabel("PC1")
    axis.set_ylabel("PC2")
    axis.legend(fontsize=8)
    plt.tight_layout()
    return plot_df


if RUN_EMBEDDING_PROBES and model is not None and tokenizer is not None and len(df):
    prompt_input_emb_df = plot_input_embedding_space(
        df.drop_duplicates("case_id"),
        text_col="prompt",
        label_col="family",
        title="Prompt input-embedding PCA by family",
    )
    completion_input_emb_df = plot_input_embedding_space(
        df,
        text_col="completion_clean",
        label_col="name",
        title="Completion input-embedding PCA by schedule",
    )
else:
    print("Skipping input embedding space plots.")


In [ ]:
def token_embedding_path(text, max_length=160):
    if model is None or tokenizer is None:
        print("Model/tokenizer unavailable.")
        return pd.DataFrame()
    try:
        emb = get_input_embedding_layer()
        dev = embedding_device(emb)
        batch = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_length).to(dev)
        with torch.inference_mode():
            token_emb = emb(batch["input_ids"])[0].float().cpu().numpy()
    except Exception as exc:
        print("Token embedding path failed:", repr(exc))
        return pd.DataFrame()

    ids = batch["input_ids"][0].detach().cpu().tolist()
    toks = tokenizer.convert_ids_to_tokens(ids)
    coords = pca_2d_embedding(token_emb)
    out = pd.DataFrame({"pos": range(len(ids)), "token_id": ids, "token": toks, "x": coords[:, 0], "y": coords[:, 1]})

    fig, axis = plt.subplots(figsize=(9, 6))
    axis.plot(out["x"], out["y"], marker="o", linewidth=1)
    for _, row in out.iloc[::max(1, len(out)//30)].iterrows():
        axis.annotate(str(row["pos"]), (row["x"], row["y"]), fontsize=8)
    axis.set_title("Token input-embedding path")
    axis.set_xlabel("PC1")
    axis.set_ylabel("PC2")
    plt.tight_layout()
    display(out.head(100))
    return out


def nearest_embedding_tokens(query, top_k=12, chunk_size=8192):
    if model is None or tokenizer is None:
        print("Model/tokenizer unavailable.")
        return pd.DataFrame()
    try:
        emb = get_input_embedding_layer(verbose=False)
        weight = emb.weight.detach()
        dev = weight.device
        ids = tokenizer(query, add_special_tokens=False, return_tensors="pt").input_ids[0].to(dev)
        if ids.numel() == 0:
            return pd.DataFrame()

        with torch.inference_mode():
            query_vec = weight.index_select(0, ids).float().mean(dim=0)
            query_vec = query_vec / query_vec.norm().clamp(min=1e-8)
            best_scores = []
            best_ids = []
            vocab_size = weight.shape[0]
            for start in range(0, vocab_size, chunk_size):
                chunk = weight[start:start + chunk_size].float()
                chunk = chunk / chunk.norm(dim=1, keepdim=True).clamp(min=1e-8)
                scores = chunk @ query_vec
                k = min(top_k, scores.numel())
                vals, idx = torch.topk(scores, k=k)
                best_scores.append(vals.cpu())
                best_ids.append((idx + start).cpu())
            scores = torch.cat(best_scores)
            token_ids = torch.cat(best_ids)
            vals, idx = torch.topk(scores, k=min(top_k, scores.numel()))
            token_ids = token_ids[idx].tolist()
            tokens = tokenizer.convert_ids_to_tokens(token_ids)
        return pd.DataFrame({"query": query, "rank": range(1, len(token_ids) + 1), "token_id": token_ids, "token": tokens, "cosine": vals.tolist()})
    except Exception as exc:
        print(f"Nearest embedding tokens failed for {query!r}:", repr(exc))
        return pd.DataFrame()


if RUN_EMBEDDING_PROBES and model is not None and tokenizer is not None and len(df):
    token_input_embedding_path_df = token_embedding_path(df.iloc[0]["prompt"])
    probe_terms = ["notebook", "change", "JSON", "double"]
    neighbor_tables = [nearest_embedding_tokens(term, top_k=10) for term in probe_terms]
    neighbor_tables = [table for table in neighbor_tables if len(table)]
    if neighbor_tables:
        neighbors = pd.concat(neighbor_tables, ignore_index=True)
        display(neighbors)
    else:
        print("No nearest-neighbor embedding tables were produced.")
else:
    print("Skipping token embedding path and nearest-neighbor probes.")


## Representation Probes

This section tries to extract hidden-state representations from the model. It may need adjustment for a different DLM repo because remote model code is not standardized.

What these plots can show:

- Do prompt variants cluster by family?
- Do completions cluster by schedule?
- Are failure cases separated from cleaner cases?
- Do answer-changing counterfactuals move in representation space?

What these plots do not prove:

- They do not prove the model performed faithful reasoning.
- They do not expose the full denoising trajectory unless the model returns it.
- PCA is only a view, not the latent space itself.


In [ ]:
def pca_2d(x):
    x = np.asarray(x, dtype=np.float32)
    if x.ndim != 2 or len(x) == 0:
        return np.zeros((0, 2), dtype=np.float32)
    x = x - x.mean(axis=0, keepdims=True)
    u, s, vt = np.linalg.svd(x, full_matrices=False)
    if vt.shape[0] == 1:
        return np.column_stack([x @ vt[0], np.zeros(len(x))])
    return x @ vt[:2].T


def _get_hidden_states(batch):
    with torch.inference_mode():
        try:
            out = model(**batch, output_hidden_states=True, return_dict=True)
        except TypeError:
            out = model(input_ids=batch["input_ids"], attention_mask=batch.get("attention_mask"), output_hidden_states=True)
    hidden = getattr(out, "hidden_states", None)
    if hidden is None and isinstance(out, dict):
        hidden = out.get("hidden_states")
    if hidden is None:
        raise RuntimeError("Model forward did not return hidden_states.")
    return hidden


def text_representations(texts, layer=-1, max_length=256, pooling="mean"):
    if model is None or tokenizer is None:
        raise RuntimeError("Model/tokenizer are not loaded.")
    batch = tokenizer(list(texts), return_tensors="pt", padding=True, truncation=True, max_length=max_length).to(device)
    hidden = _get_hidden_states(batch)[layer]
    mask = batch.get("attention_mask")
    if pooling == "last":
        lengths = mask.sum(dim=1).clamp(min=1) - 1
        reps = hidden[torch.arange(hidden.shape[0], device=hidden.device), lengths]
    else:
        mask_f = mask.unsqueeze(-1).to(hidden.dtype)
        reps = (hidden * mask_f).sum(dim=1) / mask_f.sum(dim=1).clamp(min=1)
    return reps.float().cpu().numpy()


def plot_text_representations(df, text_col="prompt", label_col="family", layer=-1, title=None):
    if df.empty:
        print("No data.")
        return None
    try:
        reps = text_representations(df[text_col].astype(str).tolist(), layer=layer)
    except Exception as exc:
        print("Representation probe failed:", repr(exc))
        return None

    coords = pca_2d(reps)
    plot_df = df[["case_id", "name", "family", "variant", "category"]].copy()
    plot_df["x"] = coords[:, 0]
    plot_df["y"] = coords[:, 1]

    fig, axis = plt.subplots(figsize=(9, 6))
    for label, group in plot_df.groupby(label_col):
        axis.scatter(group["x"], group["y"], label=label, s=70)
        for _, row in group.iterrows():
            axis.annotate(row["case_id"] if text_col == "prompt" else row["name"], (row["x"], row["y"]), fontsize=8, alpha=0.8)
    axis.set_title(title or f"PCA of {text_col} representations, layer {layer}")
    axis.set_xlabel("PC1")
    axis.set_ylabel("PC2")
    axis.legend(fontsize=8)
    plt.tight_layout()
    return plot_df

if RUN_LATENT_PROBES and model is not None and tokenizer is not None and len(df):
    prompt_rep_df = plot_text_representations(df.drop_duplicates("case_id"), text_col="prompt", label_col="family", layer=-1, title="Prompt representation PCA")
    completion_rep_df = plot_text_representations(df, text_col="completion_clean", label_col="name", layer=-1, title="Completion representation PCA by schedule")
else:
    print("Skipping representation probes.")


## Token Path Probe

This plots a single text's token hidden states in PCA space and connects them in token order. It is useful for spotting abrupt representational jumps, but remember: this is token-position structure from a forward pass, not the diffusion denoising path.


In [ ]:
def _normalize_token_hidden_tensor(hidden_tensor, expected_len=None):
    """Return a [seq, dim] numpy array from model-specific hidden-state shapes."""
    h = hidden_tensor.detach().float().cpu()

    # Common shapes: [batch, seq, dim] or [seq, batch, dim].
    if h.ndim == 3:
        if h.shape[0] == 1:
            h = h[0]
        elif h.shape[1] == 1:
            h = h[:, 0, :]
        elif expected_len is not None and h.shape[0] == expected_len:
            h = h[:, 0, :]
        elif expected_len is not None and h.shape[1] == expected_len:
            h = h[0]
        else:
            h = h.reshape(-1, h.shape[-1])
    elif h.ndim > 3:
        h = h.reshape(-1, h.shape[-1])

    if h.ndim != 2:
        raise ValueError(f"Expected normalized hidden states to be 2D [seq, dim], got shape {tuple(h.shape)}")
    return h.numpy()


def token_path_probe(text, layer=-1, max_length=160):
    if model is None or tokenizer is None:
        print("Model/tokenizer unavailable.")
        return pd.DataFrame()

    input_device = get_model_input_device() if "get_model_input_device" in globals() else device
    batch = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_length).to(input_device)
    ids = batch["input_ids"][0].detach().cpu().tolist()
    toks = tokenizer.convert_ids_to_tokens(ids)

    try:
        hidden_tensor = _get_hidden_states(batch)[layer]
        hidden = _normalize_token_hidden_tensor(hidden_tensor, expected_len=len(ids))
    except Exception as exc:
        print("Token path probe failed:", repr(exc))
        return pd.DataFrame()

    coords = pca_2d(hidden)
    n = min(len(ids), len(toks), coords.shape[0])
    if n == 0:
        print("Token path probe produced no aligned tokens.")
        return pd.DataFrame()
    if n != len(ids) or n != coords.shape[0]:
        print(
            "Token/hidden length mismatch; truncating for visualization:",
            {"tokens": len(ids), "hidden_positions": int(coords.shape[0]), "used": int(n)},
        )

    out = pd.DataFrame({
        "pos": range(n),
        "token_id": ids[:n],
        "token": toks[:n],
        "x": coords[:n, 0],
        "y": coords[:n, 1],
    })

    fig, axis = plt.subplots(figsize=(9, 6))
    axis.plot(out["x"], out["y"], marker="o", linewidth=1)
    for _, row in out.iloc[::max(1, len(out)//25)].iterrows():
        axis.annotate(str(row["pos"]), (row["x"], row["y"]), fontsize=8)
    axis.set_title(f"Token hidden-state path, layer {layer}")
    axis.set_xlabel("PC1")
    axis.set_ylabel("PC2")
    plt.tight_layout()
    display(out.head(80))
    return out

if RUN_LATENT_PROBES and model is not None and tokenizer is not None and len(df):
    token_path_df = token_path_probe(df.iloc[0]["prompt"], layer=-1)
else:
    print("Skipping token path probe.")


## High-Dimensional 3D Views

These plots project high-dimensional vectors into 3D with PCA. They are useful for quick geometric inspection, not proof of a mechanism.

What you can inspect:

- prompt clusters in input-embedding space
- completion clusters by schedule
- optional hidden-state clusters when `RUN_LATENT_PROBES=True`
- whether failures separate spatially from cleaner generations

The axes are principal components, not native model dimensions. Use the explained-variance values as a rough check for how much structure the 3D view preserves.


In [ ]:
def pca_3d(x):
    x = np.asarray(x, dtype=np.float32)
    if x.ndim != 2 or len(x) == 0:
        return np.zeros((0, 3), dtype=np.float32), np.zeros(3, dtype=np.float32)
    x = x - x.mean(axis=0, keepdims=True)
    u, s, vt = np.linalg.svd(x, full_matrices=False)
    dims = min(3, vt.shape[0])
    coords = np.zeros((x.shape[0], 3), dtype=np.float32)
    coords[:, :dims] = x @ vt[:dims].T
    denom = float((s ** 2).sum()) if len(s) else 0.0
    explained = np.zeros(3, dtype=np.float32)
    if denom > 0:
        explained[:dims] = (s[:dims] ** 2) / denom
    return coords, explained


def plot_3d_projection(plot_df, title, label_col="name", annotate_col="case_id"):
    if plot_df.empty:
        print("No 3D projection data to plot.")
        return
    fig = plt.figure(figsize=(10, 8))
    axis = fig.add_subplot(111, projection="3d")
    for label, group in plot_df.groupby(label_col):
        axis.scatter(group["pc1"], group["pc2"], group["pc3"], label=label, s=70, depthshade=True)
        for _, row in group.iterrows():
            text = str(row.get(annotate_col, ""))
            if text:
                axis.text(row["pc1"], row["pc2"], row["pc3"], text, fontsize=7)
    axis.set_title(title)
    axis.set_xlabel("PC1")
    axis.set_ylabel("PC2")
    axis.set_zlabel("PC3")
    axis.legend(fontsize=8)
    plt.tight_layout()


def build_3d_text_projection(df, reps, label_col="name", annotate_col="case_id"):
    coords, explained = pca_3d(reps)
    out = df[["case_id", "family", "variant", "category", "name"]].copy()
    out["pc1"] = coords[:, 0]
    out["pc2"] = coords[:, 1]
    out["pc3"] = coords[:, 2]
    out["label"] = out[label_col] if label_col in out else "all"
    print("3D PCA explained variance:", {"pc1": float(explained[0]), "pc2": float(explained[1]), "pc3": float(explained[2])})
    return out


def plot_text_space_3d(df, text_col="prompt", representation="input_embedding", label_col="family", title=None):
    if df.empty:
        print("No generation rows for 3D projection.")
        return pd.DataFrame()
    try:
        if representation == "input_embedding":
            reps = text_input_embedding_representations(df[text_col].astype(str).tolist())
        elif representation == "hidden":
            reps = text_representations(df[text_col].astype(str).tolist(), layer=-1)
        else:
            raise ValueError(f"Unknown representation: {representation}")
    except Exception as exc:
        print(f"3D {representation} projection failed:", repr(exc))
        return pd.DataFrame()

    plot_df = build_3d_text_projection(df, reps, label_col=label_col)
    plot_3d_projection(
        plot_df,
        title or f"3D PCA of {text_col} using {representation}",
        label_col=label_col,
        annotate_col="case_id",
    )
    return plot_df


if len(df) and model is not None and tokenizer is not None:
    if RUN_EMBEDDING_PROBES and "text_input_embedding_representations" in globals():
        prompt_input_emb_3d_df = plot_text_space_3d(
            df.drop_duplicates("case_id"),
            text_col="prompt",
            representation="input_embedding",
            label_col="family",
            title="3D PCA: prompt input embeddings by family",
        )
        completion_input_emb_3d_df = plot_text_space_3d(
            df,
            text_col="completion_clean",
            representation="input_embedding",
            label_col="name",
            title="3D PCA: completion input embeddings by schedule",
        )
    else:
        print("Skipping 3D input-embedding projection.")

    if RUN_LATENT_PROBES and "text_representations" in globals():
        prompt_hidden_3d_df = plot_text_space_3d(
            df.drop_duplicates("case_id"),
            text_col="prompt",
            representation="hidden",
            label_col="family",
            title="3D PCA: prompt hidden states by family",
        )
        completion_hidden_3d_df = plot_text_space_3d(
            df,
            text_col="completion_clean",
            representation="hidden",
            label_col="name",
            title="3D PCA: completion hidden states by schedule",
        )
    else:
        print("Skipping 3D hidden-state projection.")
else:
    print("Skipping high-dimensional 3D views; no real rows or model/tokenizer unavailable.")


## Per-Prompt Hidden-State Trajectories In 3D

This view plots each prompt as a token-position trajectory through hidden-state space. For a prompt with tokens `t0, t1, ...`, the cell extracts one layer's hidden states, projects those token vectors into 3D with PCA, and connects them in token order.

This is useful for inspecting whether a prompt has abrupt representational jumps, whether related prompts take similar paths, and whether counterfactual edits bend the path around the edited token region.

Caveats:

- This is a forward-pass hidden-state path over token positions, not the DLM denoising-time trajectory.
- The 3D axes are PCA components fitted per prompt by default, so compare shapes cautiously.
- Set `PROMPT_TRAJECTORY_CASE_IDS` to a short list if plotting every prompt gets noisy.


In [ ]:
PROMPT_TRAJECTORY_CASE_IDS = None  # Example: ["price_base", "price_cf"]
PROMPT_TRAJECTORY_LAYER = -1
PROMPT_TRAJECTORY_MAX_PROMPTS = 12
PROMPT_TRAJECTORY_MAX_LENGTH = 192


def hidden_state_token_trajectory_3d(text, layer=-1, max_length=192, title=None, annotate_every=None):
    if model is None or tokenizer is None:
        print("Model/tokenizer unavailable.")
        return pd.DataFrame()

    input_device = get_model_input_device() if "get_model_input_device" in globals() else device
    batch = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_length).to(input_device)
    ids = batch["input_ids"][0].detach().cpu().tolist()
    toks = tokenizer.convert_ids_to_tokens(ids)

    try:
        hidden_tensor = _get_hidden_states(batch)[layer]
        hidden = _normalize_token_hidden_tensor(hidden_tensor, expected_len=len(ids))
    except Exception as exc:
        print("Hidden-state trajectory failed:", repr(exc))
        return pd.DataFrame()

    coords, explained = pca_3d(hidden)
    n = min(len(ids), len(toks), coords.shape[0])
    if n == 0:
        print("Hidden-state trajectory produced no aligned tokens.")
        return pd.DataFrame()
    if n != len(ids) or n != coords.shape[0]:
        print(
            "Token/hidden length mismatch; truncating trajectory:",
            {"tokens": len(ids), "hidden_positions": int(coords.shape[0]), "used": int(n)},
        )

    out = pd.DataFrame({
        "pos": range(n),
        "token_id": ids[:n],
        "token": toks[:n],
        "pc1": coords[:n, 0],
        "pc2": coords[:n, 1],
        "pc3": coords[:n, 2],
    })

    fig = plt.figure(figsize=(10, 8))
    axis = fig.add_subplot(111, projection="3d")
    axis.plot(out["pc1"], out["pc2"], out["pc3"], linewidth=1.5, alpha=0.85)
    points = axis.scatter(
        out["pc1"], out["pc2"], out["pc3"],
        c=out["pos"], cmap="viridis", s=35, depthshade=True,
    )

    if annotate_every is None:
        annotate_every = max(1, len(out) // 20)
    for _, row in out.iloc[::annotate_every].iterrows():
        axis.text(row["pc1"], row["pc2"], row["pc3"], str(row["pos"]), fontsize=7)

    axis.set_title(title or f"Prompt hidden-state trajectory, layer {layer}")
    axis.set_xlabel("PC1")
    axis.set_ylabel("PC2")
    axis.set_zlabel("PC3")
    fig.colorbar(points, ax=axis, shrink=0.65, label="token position")
    plt.tight_layout()
    print("3D trajectory explained variance:", {"pc1": float(explained[0]), "pc2": float(explained[1]), "pc3": float(explained[2])})
    display(out.head(120))
    return out


def plot_each_prompt_hidden_trajectory_3d(
    df,
    case_ids=None,
    layer=-1,
    max_prompts=12,
    max_length=192,
):
    if df.empty:
        print("No generation rows for prompt trajectory plots.")
        return {}
    prompt_df = df.drop_duplicates("case_id")[["case_id", "family", "variant", "category", "prompt"]].copy()
    if case_ids is not None:
        wanted = set(case_ids)
        prompt_df = prompt_df[prompt_df["case_id"].isin(wanted)]
    prompt_df = prompt_df.head(max_prompts)

    trajectories = {}
    for _, row in prompt_df.iterrows():
        title = f"{row['case_id']} | {row['family']} | {row['variant']} | layer {layer}"
        print("=" * 100)
        print(title)
        trajectories[row["case_id"]] = hidden_state_token_trajectory_3d(
            row["prompt"],
            layer=layer,
            max_length=max_length,
            title=title,
        )
    return trajectories


if RUN_LATENT_PROBES and model is not None and tokenizer is not None and len(df):
    prompt_hidden_trajectory_3d = plot_each_prompt_hidden_trajectory_3d(
        df,
        case_ids=PROMPT_TRAJECTORY_CASE_IDS,
        layer=PROMPT_TRAJECTORY_LAYER,
        max_prompts=PROMPT_TRAJECTORY_MAX_PROMPTS,
        max_length=PROMPT_TRAJECTORY_MAX_LENGTH,
    )
else:
    print("Skipping per-prompt hidden-state trajectories; enable RUN_LATENT_PROBES with a loaded model and real rows.")


## Prompt Geometry: Cosine Similarity, Depth, Accuracy

This is a different kind of 3D view from PCA. Instead of projecting hidden vectors into arbitrary principal components, it makes the axes explicit:

- **x:** prompt-completion cosine similarity
- **y:** depth, using `nfe` if available, otherwise `steps`
- **z:** accuracy, using `accuracy_for_plots`

Each prompt becomes a trajectory across schedules. This lets you see whether deeper denoising moves a prompt toward or away from the prompt semantics, and whether that movement aligns with correctness.

Cosine sources:

- `input_embedding`: mean-pooled input embeddings from the model
- `hidden`: mean-pooled hidden states from the model
- `lexical`: token-count cosine over prompt/completion text
- `auto`: use input embeddings when available, otherwise lexical

For prompts without an expected answer, accuracy is unknown. Those points are plotted at `-0.1` on the z-axis so they remain visible without pretending they are correct or incorrect.


In [ ]:
PROMPT_GEOMETRY_COSINE_SOURCE = "auto"  # "auto", "input_embedding", "hidden", or "lexical"
PROMPT_GEOMETRY_DEPTH_COL = "nfe"       # "nfe" or "steps"


def lexical_vector(text):
    toks = re.findall(r"[a-zA-Z0-9$]+", normalize_text(text))
    counts = {}
    for tok in toks:
        counts[tok] = counts.get(tok, 0.0) + 1.0
    return counts


def lexical_cosine(a, b):
    va = lexical_vector(a)
    vb = lexical_vector(b)
    if not va or not vb:
        return np.nan
    keys = set(va) | set(vb)
    dot = sum(va.get(k, 0.0) * vb.get(k, 0.0) for k in keys)
    na = math.sqrt(sum(v * v for v in va.values()))
    nb = math.sqrt(sum(v * v for v in vb.values()))
    if na == 0 or nb == 0:
        return np.nan
    return dot / (na * nb)


def rowwise_cosine(a_reps, b_reps):
    a = np.asarray(a_reps, dtype=np.float32)
    b = np.asarray(b_reps, dtype=np.float32)
    denom = np.linalg.norm(a, axis=1) * np.linalg.norm(b, axis=1)
    denom = np.clip(denom, 1e-8, None)
    return np.sum(a * b, axis=1) / denom


def compute_prompt_completion_cosine(df, source="auto"):
    if df.empty:
        return np.array([], dtype=np.float32), "none"
    source_used = source
    if source == "auto":
        source_used = "input_embedding" if (RUN_EMBEDDING_PROBES and model is not None and tokenizer is not None and "text_input_embedding_representations" in globals()) else "lexical"

    prompts = df["prompt"].astype(str).tolist()
    completions = df["completion_clean"].astype(str).tolist()

    if source_used == "input_embedding":
        try:
            prompt_reps = text_input_embedding_representations(prompts)
            completion_reps = text_input_embedding_representations(completions)
            return rowwise_cosine(prompt_reps, completion_reps), source_used
        except Exception as exc:
            print("Input-embedding cosine failed; falling back to lexical cosine:", repr(exc))
            source_used = "lexical"

    if source_used == "hidden":
        try:
            prompt_reps = text_representations(prompts, layer=-1)
            completion_reps = text_representations(completions, layer=-1)
            return rowwise_cosine(prompt_reps, completion_reps), source_used
        except Exception as exc:
            print("Hidden-state cosine failed; falling back to lexical cosine:", repr(exc))
            source_used = "lexical"

    if source_used == "lexical":
        return np.array([lexical_cosine(p, c) for p, c in zip(prompts, completions)], dtype=np.float32), source_used

    raise ValueError(f"Unknown cosine source: {source}")


def build_prompt_geometry_df(df, cosine_source="auto", depth_col="nfe"):
    if df.empty:
        print("No generation rows for prompt geometry.")
        return pd.DataFrame()
    out = df.copy()
    if depth_col not in out or out[depth_col].isna().all():
        depth_col = "steps"
    if depth_col not in out:
        print("No usable depth column found.")
        return pd.DataFrame()

    cosines, source_used = compute_prompt_completion_cosine(out, source=cosine_source)
    out["prompt_completion_cosine"] = cosines
    out["cosine_source"] = source_used
    out["depth_value"] = pd.to_numeric(out[depth_col], errors="coerce")
    out["depth_source"] = depth_col
    out["accuracy_value"] = pd.to_numeric(out.get("accuracy_for_plots"), errors="coerce")
    out["accuracy_plot"] = out["accuracy_value"].fillna(-0.1)
    cols = [
        "case_id", "family", "variant", "name", "expected", "last_number_clean",
        "prompt_completion_cosine", "cosine_source", "depth_value", "depth_source",
        "accuracy_value", "accuracy_plot", "anchor_score_clean", "tail_drift_words", "special_token_count",
    ]
    return out[[c for c in cols if c in out.columns]].sort_values(["case_id", "depth_value", "name"])


def plot_prompt_geometry_3d(geometry_df, color_col="family", line_col="case_id"):
    if geometry_df.empty:
        print("No prompt geometry rows to plot.")
        return
    fig = plt.figure(figsize=(11, 8))
    axis = fig.add_subplot(111, projection="3d")

    color_values = list(geometry_df[color_col].fillna("unknown").unique()) if color_col in geometry_df else ["all"]
    cmap = plt.get_cmap("tab10")
    color_map = {value: cmap(i % 10) for i, value in enumerate(color_values)}

    for group_id, group in geometry_df.groupby(line_col):
        group = group.sort_values("depth_value")
        colors = [color_map.get(row[color_col], "black") if color_col in group else "black" for _, row in group.iterrows()]
        axis.plot(
            group["prompt_completion_cosine"],
            group["depth_value"],
            group["accuracy_plot"],
            linewidth=1.2,
            alpha=0.65,
        )
        axis.scatter(
            group["prompt_completion_cosine"],
            group["depth_value"],
            group["accuracy_plot"],
            c=colors,
            s=70,
            depthshade=True,
        )
        for _, row in group.iterrows():
            axis.text(
                row["prompt_completion_cosine"],
                row["depth_value"],
                row["accuracy_plot"],
                f"{row['case_id']}:{row['name']}",
                fontsize=7,
            )

    axis.set_xlabel("prompt-completion cosine")
    axis.set_ylabel(str(geometry_df["depth_source"].iloc[0]))
    axis.set_zlabel("accuracy (-0.1 = unknown)")
    axis.set_title(f"Prompt geometry: cosine/depth/accuracy ({geometry_df['cosine_source'].iloc[0]} cosine)")
    axis.set_zlim(-0.15, 1.05)

    handles = []
    for value, color in color_map.items():
        handles.append(plt.Line2D([0], [0], marker="o", color="w", label=value, markerfacecolor=color, markersize=8))
    if handles:
        axis.legend(handles=handles, fontsize=8, loc="upper left")
    plt.tight_layout()


def plot_prompt_geometry_2d_facets(geometry_df):
    if geometry_df.empty:
        print("No prompt geometry rows for 2D facets.")
        return
    fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))
    pairs = [
        ("depth_value", "prompt_completion_cosine", "Depth vs cosine"),
        ("depth_value", "accuracy_plot", "Depth vs accuracy"),
        ("prompt_completion_cosine", "accuracy_plot", "Cosine vs accuracy"),
    ]
    for axis, (x, y, title) in zip(axes, pairs):
        for case_id, group in geometry_df.groupby("case_id"):
            group = group.sort_values("depth_value")
            axis.plot(group[x], group[y], marker="o", label=case_id, alpha=0.75)
        axis.set_xlabel(x)
        axis.set_ylabel(y)
        axis.set_title(title)
    axes[-1].legend(fontsize=7, bbox_to_anchor=(1.05, 1), loc="upper left")
    plt.tight_layout()


def plot_prompt_geometry_plotly(geometry_df):
    try:
        import plotly.express as px
    except Exception as exc:
        print("Plotly is not installed; skipping interactive prompt geometry:", repr(exc))
        return None
    if geometry_df.empty:
        print("No prompt geometry rows for Plotly.")
        return None
    fig = px.line_3d(
        geometry_df.sort_values(["case_id", "depth_value"]),
        x="prompt_completion_cosine",
        y="depth_value",
        z="accuracy_plot",
        color="case_id",
        line_group="case_id",
        hover_data=[c for c in ["family", "variant", "name", "expected", "last_number_clean", "anchor_score_clean", "tail_drift_words", "special_token_count"] if c in geometry_df],
        title=f"Prompt trajectories in cosine-depth-accuracy space ({geometry_df['cosine_source'].iloc[0]} cosine)",
    )
    fig.show()
    return fig


if len(df):
    prompt_geometry_df = build_prompt_geometry_df(
        df,
        cosine_source=PROMPT_GEOMETRY_COSINE_SOURCE,
        depth_col=PROMPT_GEOMETRY_DEPTH_COL,
    )
    display(prompt_geometry_df)
    plot_prompt_geometry_3d(prompt_geometry_df)
    plot_prompt_geometry_2d_facets(prompt_geometry_df)
    prompt_geometry_plotly = plot_prompt_geometry_plotly(prompt_geometry_df)
else:
    print("Skipping prompt geometry; no real generation rows.")


## Similarity Matrices And Layer Sweeps

These views compare examples in representation space:

- pairwise cosine similarity matrices for prompts and completions
- optional hidden-state layer sweeps across multiple layers
- token hidden-state step-distance curves
- optional Plotly 3D scatter if Plotly is installed


In [ ]:
import plotly.express as px

def cosine_similarity_matrix(reps):
    x = np.asarray(reps, dtype=np.float32)
    if x.ndim != 2 or len(x) == 0:
        return np.zeros((0, 0), dtype=np.float32)
    x = x / np.clip(np.linalg.norm(x, axis=1, keepdims=True), 1e-8, None)
    return x @ x.T


def plot_similarity_matrix(labels, reps, title):
    sim = cosine_similarity_matrix(reps)
    if sim.size == 0:
        print("No similarity data for", title)
        return pd.DataFrame()
    fig, axis = plt.subplots(figsize=(max(7, 0.45 * len(labels)), max(6, 0.45 * len(labels))))
    im = axis.imshow(sim, vmin=-1, vmax=1, cmap="coolwarm")
    axis.set_xticks(range(len(labels)))
    axis.set_xticklabels(labels, rotation=70, ha="right", fontsize=8)
    axis.set_yticks(range(len(labels)))
    axis.set_yticklabels(labels, fontsize=8)
    axis.set_title(title)
    fig.colorbar(im, ax=axis, shrink=0.8)
    plt.tight_layout()
    return pd.DataFrame(sim, index=labels, columns=labels)


def representation_similarity_views(df):
    if df.empty or model is None or tokenizer is None:
        print("Skipping similarity matrices; no rows or model/tokenizer unavailable.")
        return {}
    outputs = {}
    labels = (df["case_id"] + ":" + df["name"]).tolist()
    prompt_df = df.drop_duplicates("case_id")
    prompt_labels = prompt_df["case_id"].tolist()

    if RUN_EMBEDDING_PROBES and "text_input_embedding_representations" in globals():
        try:
            prompt_reps = text_input_embedding_representations(prompt_df["prompt"].astype(str).tolist())
            outputs["prompt_input_embedding"] = plot_similarity_matrix(prompt_labels, prompt_reps, "Prompt input-embedding cosine similarity")
            completion_reps = text_input_embedding_representations(df["completion_clean"].astype(str).tolist())
            outputs["completion_input_embedding"] = plot_similarity_matrix(labels, completion_reps, "Completion input-embedding cosine similarity")
        except Exception as exc:
            print("Input-embedding similarity failed:", repr(exc))

    if RUN_LATENT_PROBES and "text_representations" in globals():
        try:
            prompt_hidden = text_representations(prompt_df["prompt"].astype(str).tolist(), layer=-1)
            outputs["prompt_hidden"] = plot_similarity_matrix(prompt_labels, prompt_hidden, "Prompt final-hidden cosine similarity")
            completion_hidden = text_representations(df["completion_clean"].astype(str).tolist(), layer=-1)
            outputs["completion_hidden"] = plot_similarity_matrix(labels, completion_hidden, "Completion final-hidden cosine similarity")
        except Exception as exc:
            print("Hidden-state similarity failed:", repr(exc))
    return outputs


def infer_layer_indices(num_layers, max_layers=8):
    if num_layers <= max_layers:
        return list(range(num_layers))
    return sorted(set(np.linspace(0, num_layers - 1, max_layers, dtype=int).tolist()))


def layer_sweep_projection(texts, labels, max_layers=8, max_length=256):
    if model is None or tokenizer is None:
        print("Model/tokenizer unavailable for layer sweep.")
        return pd.DataFrame()
    batch = tokenizer(list(texts), return_tensors="pt", padding=True, truncation=True, max_length=max_length).to(get_model_input_device())
    try:
        hidden_states = _get_hidden_states(batch)
    except Exception as exc:
        print("Layer sweep failed:", repr(exc))
        return pd.DataFrame()
    layer_ids = infer_layer_indices(len(hidden_states), max_layers=max_layers)
    rows = []
    for layer_id in layer_ids:
        h = hidden_states[layer_id]
        mask = batch.get("attention_mask")
        if h.ndim == 3 and h.shape[0] == mask.shape[0]:
            mask_f = mask.unsqueeze(-1).to(h.dtype)
            reps = (h * mask_f).sum(dim=1) / mask_f.sum(dim=1).clamp(min=1)
        else:
            reps = h.reshape(h.shape[0], -1, h.shape[-1]).mean(dim=1)
        coords, explained = pca_3d(reps.float().cpu().numpy())
        for label, coord in zip(labels, coords):
            rows.append({"layer": layer_id, "label": label, "pc1": coord[0], "pc2": coord[1], "pc3": coord[2], "ev3": float(explained[:3].sum())})
    out = pd.DataFrame(rows)
    if out.empty:
        return out
    fig, axis = plt.subplots(figsize=(10, 6))
    for label, group in out.groupby("label"):
        axis.plot(group["layer"], group["pc1"], marker="o", label=label)
    axis.set_xlabel("layer index")
    axis.set_ylabel("PC1 coordinate")
    axis.set_title("Layer sweep trajectory over pooled hidden states")
    axis.legend(fontsize=8)
    plt.tight_layout()
    display(out.head(80))
    return out


def hidden_step_distance_curve(text, layer=-1, max_length=192, title=None):
    if model is None or tokenizer is None:
        print("Model/tokenizer unavailable.")
        return pd.DataFrame()
    batch = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_length).to(get_model_input_device())
    ids = batch["input_ids"][0].detach().cpu().tolist()
    toks = tokenizer.convert_ids_to_tokens(ids)
    try:
        hidden = _normalize_token_hidden_tensor(_get_hidden_states(batch)[layer], expected_len=len(ids))
    except Exception as exc:
        print("Hidden step-distance curve failed:", repr(exc))
        return pd.DataFrame()
    n = min(len(ids), len(toks), hidden.shape[0])
    if n < 2:
        print("Need at least two aligned hidden states.")
        return pd.DataFrame()
    h = hidden[:n]
    distances = np.linalg.norm(np.diff(h, axis=0), axis=1)
    out = pd.DataFrame({"pos": np.arange(1, n), "token": toks[1:n], "distance_from_prev": distances})
    fig, axis = plt.subplots(figsize=(12, 4))
    axis.plot(out["pos"], out["distance_from_prev"], marker="o", linewidth=1)
    axis.set_title(title or f"Hidden-state step distance, layer {layer}")
    axis.set_xlabel("token position")
    axis.set_ylabel("L2 distance from previous token state")
    plt.tight_layout()
    display(out.sort_values("distance_from_prev", ascending=False).head(20))
    return out


def plotly_3d_projection(plot_df, title, label_col="name"):
    
    if plot_df.empty:
        print("No Plotly 3D data.")
        return None
    fig = px.scatter_3d(
        plot_df,
        x="pc1", y="pc2", z="pc3",
        color=label_col,
        hover_data=[c for c in ["case_id", "family", "variant", "category", "name"] if c in plot_df],
        title=title,
    )
    fig.show()
    return fig


if len(df):
    similarity_tables = representation_similarity_views(df)
    if RUN_LATENT_PROBES and model is not None and tokenizer is not None:
        prompt_df_for_layers = df.drop_duplicates("case_id").head(8)
        layer_sweep_df = layer_sweep_projection(prompt_df_for_layers["prompt"].tolist(), prompt_df_for_layers["case_id"].tolist())
        step_distance_df = hidden_step_distance_curve(prompt_df_for_layers.iloc[0]["prompt"], title=f"Step distances: {prompt_df_for_layers.iloc[0]['case_id']}")
    else:
        print("Skipping hidden layer sweep and step-distance curves; RUN_LATENT_PROBES is disabled or model unavailable.")

    if "completion_input_emb_3d_df" in globals() and isinstance(completion_input_emb_3d_df, pd.DataFrame) and len(completion_input_emb_3d_df):
        plotly_completion_3d = plotly_3d_projection(completion_input_emb_3d_df, "Interactive 3D completion input-embedding PCA", label_col="name")
else:
    print("Skipping similarity/layer visualizations; no real generation rows.")


## Export Results

The CSV is convenient for quick analysis. JSONL preserves list-like fields such as `anchor_terms` more naturally.


In [ ]:
if SAVE_RESULTS and len(df):
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    csv_path = RUN_DIR / f"dlm_behavior_{stamp}.csv"
    jsonl_path = RUN_DIR / f"dlm_behavior_{stamp}.jsonl"
    df.to_csv(csv_path, index=False)
    with jsonl_path.open("w") as f:
        for row in df.to_dict(orient="records"):
            f.write(json.dumps(row, default=str) + "\n")
    print("saved:", csv_path)
    print("saved:", jsonl_path)
else:
    print("Nothing saved.")


## Where To Extend This Notebook

Good next additions:

1. Add a larger prompt suite with tags for skill, difficulty, ambiguity, and expected invariances.
2. Add repeated seeds per schedule to separate stable behavior from one-off runs.
3. Add exact parsers for JSON, math answers, code blocks, or theorem-proof structure.
4. Patch the model's remote `generate` method to return intermediate denoising states.
5. Compare hidden-state probes across layers, not just the final layer.
6. Cluster failure cases and then read the raw outputs for each cluster.
7. Add activation patching or logit-lens style probes if the model architecture exposes compatible internals.

The strongest workflow is: run broad sweeps, find weird clusters, inspect raw text, then build a smaller targeted mechanistic probe around the behavior you actually observe.
